# Milestone 1 – BIWI dataset check
**Screen Privacy Through Head Orientation · DMET 901**

This notebook shows the data is real and usable:
1. downloads and **loads** the dataset
2. prints the **class distribution** (forward / away) and plots it
3. shows **sample images** from each class

Run each cell from top to bottom (**Runtime → Run all**).

## 1. Download the data
Face crops from the FSA-Net project (Google Drive) and the HeadPose+ angle files (GitHub).

In [ ]:
!pip -q install gdown
# FSA-Net preprocessed data (contains BIWI_noTrack.npz). This zip is large; it can take a few minutes.
!gdown 1j6GMx33DCcbUOS8J3NHZ-BMHgk7H-oC_ -O data.zip
# Extract only the file we need
!unzip -o -j data.zip "*BIWI_noTrack.npz" -d biwi
# HeadPose+ repository (calibrated angle files)
!git clone -q https://github.com/kuhnkeF/headposeplus
!ls -lh biwi
!find headposeplus -name "*.npz"

## 2. Load the face crops

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

d = np.load("biwi/BIWI_noTrack.npz")
print("Keys in the file:", list(d.keys()))

images = d["image"]          # face crops, stored in BGR colour order
pose = d["pose"]             # one row per image: yaw, pitch, roll (degrees)

print("Images:", images.shape, images.dtype)   # (number of images, height, width, colours)
print("Angles:", pose.shape)
yaw, pitch, roll = pose[:, 0], pose[:, 1], pose[:, 2]
print(f"Yaw range:   {yaw.min():.1f}° to {yaw.max():.1f}°")
print(f"Pitch range: {pitch.min():.1f}° to {pitch.max():.1f}°")

## 3. Make the labels (our 20° rule)
**Forward** if |yaw| ≤ 20° and |pitch| ≤ 20°, otherwise **away**.

In [ ]:
THRESHOLD = 20
forward = (np.abs(yaw) <= THRESHOLD) & (np.abs(pitch) <= THRESHOLD)
labels = np.where(forward, "Forward", "Away")

n = len(labels)
n_fwd = int(forward.sum()); n_away = n - n_fwd
print(f"Total images: {n:,}")
print(f"Forward: {n_fwd:,} ({n_fwd/n:.1%})")
print(f"Away:    {n_away:,} ({n_away/n:.1%})")

print("\nHow the threshold changes the forward count:")
for t in (15, 20, 25):
    c = int(((np.abs(yaw) <= t) & (np.abs(pitch) <= t)).sum())
    print(f"  {t}° -> {c:,} forward images")

## 4. Class distribution plot

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(["Forward", "Away"], [n_fwd, n_away], color=["#1F5F84", "#E0A56E"])
for b, v in zip(bars, [n_fwd, n_away]):
    ax.text(b.get_x() + b.get_width()/2, v, f"{v:,}\n({v/n:.1%})", ha="center", va="bottom")
ax.set_ylabel("Number of images")
ax.set_title("BIWI class distribution (20° rule)")
ax.set_ylim(0, n_away * 1.2)
plt.tight_layout(); plt.savefig("class_distribution.png", dpi=200); plt.show()

## 5. Sample images from each class
The files store colours as BGR, so we flip them to RGB before showing.

In [ ]:
rng = np.random.default_rng(42)
fwd_idx = rng.choice(np.where(forward)[0], 3, replace=False)
away_idx = rng.choice(np.where(~forward)[0], 3, replace=False)

fig, axes = plt.subplots(2, 3, figsize=(9, 6.5))
for row, (name, idxs) in enumerate([("FORWARD", fwd_idx), ("AWAY", away_idx)]):
    for col, i in enumerate(idxs):
        axes[row, col].imshow(images[i][..., ::-1])      # BGR -> RGB
        axes[row, col].set_title(f"{name}\nyaw {yaw[i]:+.0f}°  pitch {pitch[i]:+.0f}°", fontsize=10)
        axes[row, col].axis("off")
plt.tight_layout(); plt.savefig("sample_images.png", dpi=200); plt.show()

## 6. Where the images fall (yaw vs. pitch)
Each dot is one image. Inside the box = forward.

In [ ]:
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(yaw[~forward], pitch[~forward], s=2, alpha=0.3, color="#E0A56E", label="Away")
ax.scatter(yaw[forward], pitch[forward], s=2, alpha=0.3, color="#1F5F84", label="Forward")
ax.add_patch(plt.Rectangle((-20, -20), 40, 40, fill=False, lw=2, color="black"))
ax.set_xlabel("Yaw (left / right, degrees)"); ax.set_ylabel("Pitch (up / down, degrees)")
ax.legend(markerscale=6); ax.set_title("Head angles and the 20° box")
plt.tight_layout(); plt.savefig("angle_scatter.png", dpi=200); plt.show()

## 7. Look inside the HeadPose+ files
This only prints what the files contain, so we can match the calibrated angles later.

In [ ]:
import glob
for path in sorted(glob.glob("headposeplus/**/*.npz", recursive=True)):
    f = np.load(path, allow_pickle=True)
    print("==", path)
    for k in f.keys():
        a = f[k]
        print(f"   {k}: shape={a.shape}, dtype={a.dtype}, first={a.ravel()[:3] if a.size else a}")

## Note on the numbers
These counts use the angles stored inside `BIWI_noTrack.npz`. Our slides use the calibrated angles from HeadPose+, which give **4,270 forward / 8,949 away**. If the numbers above differ slightly, that is why.